# Radical-Aligned Structure — Full reviewer-proofing run (10 models, expanded cloze, non-parametric stats)

Free Colab T4. ~75 minutes total — Colab cache wipes between sessions, so this is a full extraction.

**What's new vs. the last run:**
- 2 more models: MacBERT-base + ERNIE-3.0-base-zh (so the 'Chinese-specialized' category has n=4 distinct architectures, not 3)
- Expanded cloze probe: 21 semantic fields × 6–7 candidates = ~250 trial pairs (was 40)
- New non-parametric statistics: Wendt's r, character-clustered bootstrap, predicted-vs-observed Spearman ρ
- Variance decomposition over 10 models (was 8)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Run cells top to bottom. The final cell packages and downloads the new zip.

In [ ]:
# 1. Clone the repo (pulls latest with MacBERT/ERNIE in fast subset, expanded cloze, nonparametric_stats)
!git clone https://github.com/aryan35790jp/chinese_llm.git /content/repo
%cd /content/repo

In [ ]:
# 2. Install dependency stack
!pip install -q transformers==4.44.2 tokenizers==0.19.1 \
    numpy==1.26.4 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.1 \
    matplotlib==3.9.0 tqdm==4.66.4 datasets==2.20.0 sentencepiece==0.2.0 \
    Pillow==10.4.0 umap-learn==0.5.6 \
    fugashi==1.3.2 unidic-lite==1.0.8 ipadic==1.0.0

In [ ]:
# 3. Build dataset (fast — Unihan download + filter, ~3 min)
!mkdir -p data/unihan
!curl -sL -o data/unihan/Unihan.zip https://www.unicode.org/Public/UCD/latest/ucd/Unihan.zip
!cd data/unihan && unzip -o -q Unihan.zip && cd ../..
!python scripts/new/dataset_builder.py
!python scripts/new/classify_liushu.py

In [ ]:
# 4. Extract embeddings — only the new MacBERT and ERNIE will actually extract;
#    the 8 existing models are skipped via cache check.
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/extract_embeddings.py

In [ ]:
# 5. Re-run isotropy correction so MacBERT/ERNIE get corrected
!python scripts/new/isotropy_correction.py

In [ ]:
# 6. Glyph baseline (already cached, will skip)
!apt-get install -y -q fonts-noto-cjk 2>/dev/null || true
!python scripts/new/glyph_only_baseline.py

In [ ]:
# 7. Force-rerun the analyses that need to include MacBERT and ERNIE.
#    Delete stale CSVs first so full_pipeline doesn't skip them.
import os, pathlib
for f in [
    'layer_wise.csv', 'expanded_semantic_control.csv',
    'expanded_semantic_control_pooled.csv', 'probing.csv',
    'phonetic_vs_semantic_radicals.csv', 'cross_script_japanese.csv',
    'glyph_comparison.csv', 'scaling.csv', 'orthographic_arithmetic.csv',
    'orthographic_arithmetic_summary.csv', 'activation_patching.csv',
    'pseudoradical_control.csv', 'frequency_matched.csv',
    'downstream_validation.csv', 'downstream_per_radical.csv',
    'tokenization_audit.csv', 'tokenization_audit_summary.csv',
    '_REPORT.md',
]:
    p = pathlib.Path('results') / f
    if p.exists():
        p.unlink()
        print('removed', p)
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/full_pipeline.py --fast --skip extract,glyph_baseline,isotropy,cooccurrence,sentential

In [ ]:
# 8. Variance decomposition (cooccurrence_baseline) — re-run to include MacBERT and ERNIE.
#    PMI matrix is cached so this is fast (~2 min) — only the regression part runs.
import pathlib
p = pathlib.Path('results') / 'variance_decomposition.csv'
if p.exists(): p.unlink()
import os
os.environ['RADICAL_FAST'] = '1'
!RADICAL_FAST=1 python scripts/new/cooccurrence_baseline.py

In [ ]:
# 9. Non-parametric stats (Wendt's r, char-clustered bootstrap, predicted-vs-observed Spearman)
!python scripts/new/nonparametric_stats.py

# Cluster-robust regression — fixes the pair non-independence problem.
# Two-way Cameron-Gelbach-Miller cluster-robust SE on (i, j) characters.
!python scripts/new/cluster_robust_regression.py

In [ ]:
# 10. Cloze probe — generate procedural items first, then run.
import pathlib
for f in ['radical_cloze.csv', 'radical_cloze_summary.csv']:
    p = pathlib.Path('results') / f
    if p.exists(): p.unlink()
!python scripts/new/cloze_procedural.py
!python scripts/new/radical_cloze_probe.py

In [ ]:
# 11. Regenerate figures and report
!python scripts/new/figures.py
!python scripts/new/results_report.py
with open('results/_REPORT.md', encoding='utf-8') as f:
    from IPython.display import Markdown
    md = f.read()
Markdown(md)

In [ ]:
# 12. Pack and download
!cd /content/repo && zip -qr /content/results.zip results figures data/radical_dataset.csv data/radical_summary.csv
from google.colab import files
files.download('/content/results.zip')